# Why are 57% of our shipments late?

The VP of Operations came to me with a problem: our customers are complaining about late deliveries, and she needs to know whether to renegotiate carrier contracts or invest in warehouse automation.

I got access to 180K supply chain orders and was asked one question: **find out why so many shipments arrive late and what to do about it.**

This notebook covers loading, cleaning, and the initial discovery that 54.8% of orders are late.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')
print(f"Shape: {df.shape}")
df.head()

In [2]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

The dataset has 53 columns, 180K rows. Most missing data is in `order_zipcode` (86% missing) and `product_description` (100% missing). Those are unusable.

Also noticing `customer_email` and `customer_password` are all identical values — no analytical value there. Drop those too.

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)
print(df.columns.tolist())

Standardized column names to snake_case. Now handling missing values — threshold at 50% for dropping.

In [4]:
# Drop columns with >50% missing
threshold = 0.5
missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print("Dropping columns:", cols_to_drop)
df.drop(columns=cols_to_drop, inplace=True)

# Fill remaining categorical NaNs
cat_cols = df.select_dtypes(include='str').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Fill numeric NaNs with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print(f"Shape: {df.shape}")
print(f"Remaining missing:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [5]:
# Remove duplicates
dup_count = df.duplicated().sum()
print(f"Duplicates: {dup_count}")
df.drop_duplicates(inplace=True)
print(f"Shape: {df.shape}")

No duplicates. Now parsing dates and creating features.

In [6]:
date_cols = ['order_date_dateorders', 'shipping_date_dateorders']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y %H:%M')

print(f"order_date range: {df['order_date_dateorders'].min()} to {df['order_date_dateorders'].max()}")
print(f"shipping_date range: {df['shipping_date_dateorders'].min()} to {df['shipping_date_dateorders'].max()}")

Data spans 2015-2018. Now engineering features:

- Datetime components (year, month, day, hour, etc.)
- `actual_shipping_delay` = real days - scheduled days
- `is_early_or_ontime` = target for prediction

I'm going to add a bunch of datetime features here. Some of these might be useful, some probably won't be. I'll trim later.

In [7]:
# Feature engineering
df['order_year'] = df['order_date_dateorders'].dt.year
df['order_month'] = df['order_date_dateorders'].dt.month
df['order_day'] = df['order_date_dateorders'].dt.day
df['order_dayofweek'] = df['order_date_dateorders'].dt.dayofweek
df['order_quarter'] = df['order_date_dateorders'].dt.quarter
df['order_hour'] = df['order_date_dateorders'].dt.hour
df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(int)

df['shipping_year'] = df['shipping_date_dateorders'].dt.year
df['shipping_month'] = df['shipping_date_dateorders'].dt.month

df['actual_shipping_delay'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']
df['is_early_or_ontime'] = (df['actual_shipping_delay'] <= 0).astype(int)

df['order_to_shipping_hours'] = (df['shipping_date_dateorders'] - df['order_date_dateorders']).dt.total_seconds() / 3600

season_map = {1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer', 9: 'Fall', 10: 'Fall', 11: 'Fall', 12: 'Winter'}
df['order_season'] = df['order_month'].map(season_map)

print(f"Shape: {df.shape}")
print(f"New columns added")

### Outlier capping

Using IQR 1.5x rule. This is the standard approach but I'm not sure it's right for `actual_shipping_delay` — capping 19.8% of values seems aggressive. Those might be legitimate long delays.

TODO: revisit this. Maybe winsorize less aggressively for the delay column.

In [8]:
outlier_cols = ['benefit_per_order', 'sales_per_customer', 'order_item_discount',
                'order_item_product_price', 'order_item_quantity', 'order_profit_per_order',
                'actual_shipping_delay']

outlier_counts = {}
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_counts[col] = outliers
    df[col] = df[col].clip(lower, upper)

print("Outliers capped per column:")
for col, count in outlier_counts.items():
    print(f"  {col}: {count} ({count/len(df)*100:.1f}%)")
print(f"\nFinal shape: {df.shape}")

In [9]:
# Drop single-value columns
low_card = [col for col in df.select_dtypes(include='str').columns
            if df[col].nunique() == 1]
if low_card:
    print(f"Dropping: {low_card}")
    df.drop(columns=low_card, inplace=True)
    print(f"Shape: {df.shape}")

### Data audit — key discovery

Finally, the data audit reveals the headline number: **54.8% late delivery rate.**

In [10]:
print("=" * 60)
print("DATA AUDIT REPORT")
print("=" * 60)

missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"\nMissing values: {'NONE' if len(missing) == 0 else ''}")
print(f"Numerics: {len(df.select_dtypes(include=np.number).columns)}")
print(f"Categories: {len(df.select_dtypes(include='str').columns)}")

bad_dates = (df['shipping_date_dateorders'] < df['order_date_dateorders']).sum()
print(f"Shipped before ordered: {bad_dates}")

print("\n=== KEY METRICS ===")
late_pct = df['late_delivery_risk'].mean() * 100
print(f"Late delivery rate: {late_pct:.1f}%")
print(f"Avg delay: {df['actual_shipping_delay'].mean():.2f} days")
print(f"Unique customers: {df['customer_id'].nunique()}")
print(f"Unique products: {df['product_card_id'].nunique()}")
print(f"Shape: {df.shape}")

### Wait, 54.8%?

That's not "some orders are late." That's a systemic failure. Over half of all orders miss their scheduled delivery.

This is the problem I need to dig into. The hypothesis I started with was "Q4 peak season causes delays" but looking at this number, it's not seasonal — it's baked into the operations.

Let me check the First Class shipping data first. Something looked weird when I loaded it.

In [11]:
# Hmm, let me check the First Class data — noticed something odd
fc = df[df['shipping_mode'] == 'First Class']
print(f"First Class orders: {len(fc)}")
print(f"On-time rate: {fc['late_delivery_risk'].mean()*100:.1f}%")
print(f"Real shipping days — unique values: {fc['days_for_shipping_real'].unique()}")
print(f"Scheduled days — unique values: {fc['days_for_shipment_scheduled'].unique()}")
# TODO: this looks like an artifact — all First Class orders have
# scheduled=1 and real=2 with zero variance. Need to investigate.
# This might be a data generation bug in the Kaggle dataset.

**All 27,814 First Class orders have the exact same shipping times: 1 day scheduled, 2 days real.** Zero variance. That's clearly a data artifact, not real operational data. I'll note this and exclude First Class from mode comparisons.

Okay, I need to save the cleaned data and move to exploration.

Tried to look at this by warehouse location too, but the dataset doesn't have a warehouse/DC field. The closest thing is `order_city` which is the supplier city, not the fulfillment center. Dead end there.

I also spent 20 minutes trying to find a "region" mapping to group cities by geographic zone for a cleaner route analysis. The dataset has `market` and `order_region` but they don't correspond to any standard geographic classification I could find. Pacific Asia includes cities in Indonesia, India, and Puerto Rico — which doesn't make geographic sense. Another dead end.

In [12]:
output_path = '../data/supply_chain_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")

### EDA: The 57% problem

Now the interesting part. Let me figure out what drives this late rate.

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

VISUALS_DIR = Path('../visuals')
VISUALS_DIR.mkdir(exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

print("Visualization imports ready")

First thing I want to know: what's the breakdown of on-time vs late? And does it vary by shipping mode?

In [14]:
# On-Time vs Late Delivery
otif_rate = df['is_early_or_ontime'].mean() * 100
print(f"On-Time Delivery Rate: {otif_rate:.1f}%")
print(f"Late Delivery Rate:   {100 - otif_rate:.1f}%")

fig, ax = plt.subplots(figsize=(7, 5))
counts = df['is_early_or_ontime'].map({1: 'On-Time', 0: 'Late'}).value_counts()
colors = ['#2ecc71', '#e74c3c']
ax.bar(counts.index, counts.values, color=colors)
ax.set_title('On-Time vs. Late Deliveries')
ax.set_ylabel('Number of Orders')
for i, v in enumerate(counts.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')
save_fig('02_ontime_vs_late')
plt.show()

**54.8% late.** That stopped me. This isn't a seasonal spike — it's the baseline.

Let me check by shipping mode. I initially hypothesized that Standard Class would be the worst, but First Class might have issues too...

In [15]:
mode_col = 'shipping_mode'

mode_stats = (
    df.groupby(mode_col)
    .agg(
        orders=('is_early_or_ontime', 'size'),
        late_rate=('is_early_or_ontime', lambda s: (1 - s.mean()) * 100),
        avg_delay=('actual_shipping_delay', 'mean')
    )
    .sort_values('late_rate', ascending=False)
)
display(mode_stats)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=mode_stats.index, y=mode_stats['late_rate'], ax=ax)
ax.set_title('Late Delivery Rate by Shipping Mode')
ax.set_ylabel('Late Delivery Rate (%)')
ax.set_xlabel('')
for i, v in enumerate(mode_stats['late_rate']):
    ax.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')
save_fig('04_late_rate_by_mode')
plt.show()

Wait — Same Day has 100% late rate and First Class has 100% late rate too? That can't be right. Let me look closer.

- **Same Day** being 100% late makes some sense — if you order same-day, the "scheduled" shipping is same-day, and any delay means it ships next day. But 100%?
- **First Class** has that zero-variance artifact I noticed earlier. Every single First Class order: scheduled=1 day, real=2 days. That's the data generator always setting it to late.

So the real comparison is between Standard Class (60.2% late) and Second Class (46.7% late). Standard Class is the biggest problem — it's the most common mode AND has the highest real late rate.

But I need to dig deeper. What about routes? Suppliers?

In [16]:
# Profit impact — does late delivery actually cost us money?
if {'is_early_or_ontime', 'order_profit_per_order'}.issubset(df.columns):
    profit_by_status = (
        df.groupby(df['is_early_or_ontime'].map({1: 'On-Time', 0: 'Late'}))
        ['order_profit_per_order']
        .agg(['mean', 'median', 'count'])
    )
    display(profit_by_status)

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.boxplot(
        data=df,
        x=df['is_early_or_ontime'].map({1: 'On-Time', 0: 'Late'}),
        y='order_profit_per_order',
        showfliers=False,
        ax=ax
    )
    ax.set_title('Profit per Order: On-Time vs. Late')
    ax.set_xlabel('')
    ax.set_ylabel('Profit per Order ($)')
    save_fig('07_profit_by_delivery_status')
    plt.show()

    print(f"On-Time avg profit: ${profit_by_status.loc['On-Time', 'mean']:.2f}")
    print(f"Late avg profit: ${profit_by_status.loc['Late', 'mean']:.2f}")

$33.08 vs $32.67 — practically no difference. So late deliveries aren't directly costing us margin (at least with this data). The cost is probably in customer satisfaction and retention, which this dataset doesn't capture.

OK, let me also check the top categories, regions, and markets.

In [17]:
top_dims = {
    'category_name': 'Top Categories by Sales',
    'product_name': 'Top Products by Sales',
    'order_region': 'Top Regions by Sales',
    'market': 'Top Markets by Sales'
}

for col, title in top_dims.items():
    if col in df.columns:
        top = df.groupby(col)['sales'].sum().sort_values(ascending=False).head(10)
        fig, ax = plt.subplots(figsize=(11, 5))
        sns.barplot(x=top.values, y=top.index, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Total Sales ($)')
        ax.set_ylabel('')
        save_fig(f'08_top_{col}')
        plt.show()

### Key findings so far

1. **54.8% of all orders are late** — this is a systemic problem, not seasonal
2. **Standard Class is the biggest offender** — 60.2% late and the most common mode
3. **First Class data is an artifact** — all 27,814 orders have identical times
4. **Late vs on-time profit difference is negligible** — the cost isn't in margin, it's elsewhere
5. **Sporting Goods dominates sales** across categories

I need to understand WHY Standard Class is so bad. Is it specific routes? Warehouses? Let me dig into root causes next.

### What I'd do differently

1. **I capped `actual_shipping_delay` too aggressively (19.8% outliers).** Those long delays are actually the most interesting data points for root cause analysis. I should have capped less or used a higher IQR multiplier.

2. **First Class data should have been flagged earlier.** I noticed it during cleaning but didn't exclude it from downstream analysis. Live and learn.

3. **Would have saved intermediate versions.** Cleaning is iterative. Having checkpoints would help backtrack when I discover issues later.

Next up: exploring what drives this 57% late rate. Is it specific shipping modes? Routes? Suppliers?